In [ ]:
"""
Final polished USCRN mini-project workflow with ERA5 comparison support.

What this script does
---------------------
1. Reads five USCRN station-variable Excel files.
2. Merges station observations with station coordinates.
3. Computes overall, station-wise, yearly, and seasonal statistics.
4. Performs Kolmogorov-Smirnov seasonal-vs-all distribution tests.
5. Builds a 2-degree daily interpolated grid from station data.
6. Computes leave-one-out interpolation skill (RMSE, correlation).
7. Generates plots and sample maps.
8. Optionally compares station-derived daily values with ERA5/ERA5-Land point extractions.

ERA5 comparison input
---------------------
If you have extracted ERA5 data, place CSV files in a folder named:
    era5_data/
Expected format per variable (flexible column names handled where possible):
    date, StationName, era5_value
or
    date, station, value

Recommended filenames:
    ERA5_AirTemperature.csv
    ERA5_Precipitation.csv
    ERA5_RelativeHumidity.csv
    ERA5_SoilTemperature.csv
    ERA5_SoilMoisture10cm.csv

Notes
-----
- ERA5 daily 2m air temperature is typically in Kelvin.
- ERA5 daily total precipitation is typically in metres and often needs conversion to mm.
- ERA5 relative humidity is often not native in daily products and may be derived from
  2m temperature and 2m dewpoint temperature.
- Soil variables are often more appropriate from ERA5-Land than ERA5.
"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import griddata
from scipy.stats import ks_2samp

warnings.filterwarnings("ignore", category=RuntimeWarning)


# ======================================
# Paths and configuration
# ======================================
BASE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path('.')
OUTPUT_DIR = BASE_DIR / 'output'
CSV_DIR = OUTPUT_DIR / 'csv'
PLOT_DIR = OUTPUT_DIR / 'plots'
MAP_DIR = OUTPUT_DIR / 'maps'
ERA5_DIR = BASE_DIR / 'era5_data'

for folder in [OUTPUT_DIR, CSV_DIR, PLOT_DIR, MAP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

STATION_INFO_FILE = BASE_DIR / 'All_CONUS_Station_Information_V2.xlsx'

VARIABLE_CONFIGS = {
    'AirTemperature': {
        'file': BASE_DIR / 'USCRN_AirTemperature_2006_2021.xlsx',
        'unit': 'K',
        'sample_date': '2012-07-15',
        'era5_file': ERA5_DIR / 'ERA5_AirTemperature.csv',
        'era5_unit_note': 'ERA5 daily 2m temperature is commonly in K.'
    },
    'Precipitation': {
        'file': BASE_DIR / 'USCRN_Precipitation_2006_2021.xlsx',
        'unit': 'mm',
        'sample_date': '2012-07-15',
        'era5_file': ERA5_DIR / 'ERA5_Precipitation.csv',
        'era5_unit_note': 'ERA5 daily total precipitation is commonly in m and should often be converted to mm.'
    },
    'RelativeHumidity': {
        'file': BASE_DIR / 'USCRN_RelativeHumidity_2006_2021.xlsx',
        'unit': '%',
        'sample_date': '2012-07-15',
        'era5_file': ERA5_DIR / 'ERA5_RelativeHumidity.csv',
        'era5_unit_note': 'ERA5 RH is often derived from temperature and dewpoint.'
    },
    'SoilMoisture10cm': {
        'file': BASE_DIR / 'USCRN_SoilMoisture10cm_2006_2021.xlsx',
        'unit': 'm3/m3',
        'sample_date': '2012-07-15',
        'era5_file': ERA5_DIR / 'ERA5_SoilMoisture10cm.csv',
        'era5_unit_note': 'Use ERA5-Land for soil moisture where possible.'
    },
    'SoilTemperature': {
        'file': BASE_DIR / 'USCRN_SoilTemperature_2006_2021.xlsx',
        'unit': 'K',
        'sample_date': '2012-07-15',
        'era5_file': ERA5_DIR / 'ERA5_SoilTemperature.csv',
        'era5_unit_note': 'Use ERA5-Land for soil temperature where possible.'
    }
}

GRID_RES = 2.0
LON_MIN, LON_MAX = -125.0, -66.0
LAT_MIN, LAT_MAX = 24.0, 50.0
MIN_STATIONS_FOR_INTERP = 3
SKILL_SAMPLE_DATES = 60


# ======================================
# Utilities
# ======================================
def get_season(month: int) -> str:
    if month in [12, 1, 2]:
        return 'DJF'
    if month in [3, 4, 5]:
        return 'MAM'
    if month in [6, 7, 8]:
        return 'JJA'
    return 'SON'


def compute_rmse(obs: np.ndarray, pred: np.ndarray) -> float:
    mask = np.isfinite(obs) & np.isfinite(pred)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((obs[mask] - pred[mask]) ** 2)))


def compute_corr(obs: np.ndarray, pred: np.ndarray) -> float:
    mask = np.isfinite(obs) & np.isfinite(pred)
    if mask.sum() < 2:
        return np.nan
    return float(np.corrcoef(obs[mask], pred[mask])[0, 1])


def normalize_era5_columns(df: pd.DataFrame) -> pd.DataFrame:
    colmap = {c.lower().strip(): c for c in df.columns}

    date_col = None
    for c in ['date', 'time', 'datetime']:
        if c in colmap:
            date_col = colmap[c]
            break

    station_col = None
    for c in ['stationname', 'station', 'station_name']:
        if c in colmap:
            station_col = colmap[c]
            break

    value_col = None
    for c in ['era5_value', 'value', 'era5', 'reference_value']:
        if c in colmap:
            value_col = colmap[c]
            break

    if date_col is None or station_col is None or value_col is None:
        raise ValueError('ERA5 file must contain date, station/stationname, and value/era5_value columns.')

    out = df.rename(columns={date_col: 'date', station_col: 'StationName', value_col: 'era5_value'}).copy()
    out['date'] = pd.to_datetime(out['date'])
    return out[['date', 'StationName', 'era5_value']]


# ======================================
# Data loading
# ======================================
def load_station_info(path: Path) -> pd.DataFrame:
    info = pd.read_excel(path)
    return info[['StationName', 'StationID', 'Lon', 'Lat']].dropna()


def load_variable_data(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path)
    df['date'] = pd.to_datetime(df['date'])
    return df


def reshape_to_long(df_wide: pd.DataFrame) -> pd.DataFrame:
    station_cols = [c for c in df_wide.columns if c != 'date']
    df_long = df_wide.melt(id_vars='date', value_vars=station_cols,
                           var_name='StationName', value_name='value')
    return df_long.dropna(subset=['value'])


def merge_station_coords(df_long: pd.DataFrame, station_info: pd.DataFrame) -> pd.DataFrame:
    df = df_long.merge(station_info, on='StationName', how='inner')
    df = df.dropna(subset=['Lon', 'Lat', 'value'])
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['season'] = df['month'].map(get_season)
    return df


def load_era5_if_available(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return normalize_era5_columns(df)


# ======================================
# Grid and interpolation
# ======================================
def create_grid():
    lons = np.arange(LON_MIN, LON_MAX + GRID_RES, GRID_RES)
    lats = np.arange(LAT_MIN, LAT_MAX + GRID_RES, GRID_RES)
    grid_lon, grid_lat = np.meshgrid(lons, lats)
    return lons, lats, grid_lon, grid_lat


def interpolate_points(df_day: pd.DataFrame, grid_lon: np.ndarray, grid_lat: np.ndarray,
                       method: str = 'linear') -> np.ndarray:
    if len(df_day) < MIN_STATIONS_FOR_INTERP:
        return np.full(grid_lon.shape, np.nan)

    points = df_day[['Lon', 'Lat']].to_numpy()
    values = df_day['value'].to_numpy()
    grid = griddata(points, values, (grid_lon, grid_lat), method=method)

    if method != 'nearest' and np.isnan(grid).all():
        grid = griddata(points, values, (grid_lon, grid_lat), method='nearest')

    return grid


# ======================================
# Statistics
# ======================================
def descriptive_stats(df: pd.DataFrame) -> pd.DataFrame:
    vals = df['value'].dropna()
    return pd.DataFrame([{
        'count': vals.count(),
        'mean': vals.mean(),
        'median': vals.median(),
        'std': vals.std(),
        'variance': vals.var(),
        'min': vals.min(),
        'max': vals.max()
    }])


def station_stats(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby('StationName')['value'].agg(
        count='count', mean='mean', median='median', std='std',
        variance='var', min='min', max='max'
    ).reset_index()


def seasonal_stats(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby('season')['value'].agg(
        count='count', mean='mean', median='median', std='std',
        variance='var', min='min', max='max'
    ).reset_index()


def yearly_stats(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby('year')['value'].agg(
        count='count', mean='mean', median='median', std='std'
    ).reset_index()


def seasonal_distribution_tests(df: pd.DataFrame) -> pd.DataFrame:
    annual_vals = df['value'].dropna()
    rows = []
    for season, grp in df.groupby('season'):
        vals = grp['value'].dropna()
        if len(vals) > 1 and len(annual_vals) > 1:
            ks_stat, p_val = ks_2samp(vals, annual_vals)
        else:
            ks_stat, p_val = np.nan, np.nan
        rows.append({'season': season, 'ks_statistic_vs_all': ks_stat, 'p_value_vs_all': p_val})
    return pd.DataFrame(rows)


def interpolation_skill_loo(df: pd.DataFrame, method: str = 'linear', max_dates: int = 60) -> pd.DataFrame:
    available_dates = sorted(df['date'].drop_duplicates())
    if len(available_dates) == 0:
        return pd.DataFrame(columns=['date', 'n_points', 'rmse', 'corr'])

    if len(available_dates) > max_dates:
        idx = np.linspace(0, len(available_dates) - 1, max_dates, dtype=int)
        sampled_dates = [available_dates[i] for i in idx]
    else:
        sampled_dates = available_dates

    rows = []
    for dt in sampled_dates:
        day = df[df['date'] == dt].dropna(subset=['Lon', 'Lat', 'value']).copy()
        n = len(day)
        if n < 5:
            rows.append({'date': dt, 'n_points': n, 'rmse': np.nan, 'corr': np.nan})
            continue

        obs_vals = []
        pred_vals = []
        for i in range(n):
            train = day.drop(day.index[i])
            test = day.iloc[i]
            if len(train) < MIN_STATIONS_FOR_INTERP:
                continue

            pred = griddata(train[['Lon', 'Lat']].to_numpy(), train['value'].to_numpy(),
                            np.array([[test['Lon'], test['Lat']]]), method=method)
            if pred is None or len(pred) == 0 or np.isnan(pred[0]):
                pred = griddata(train[['Lon', 'Lat']].to_numpy(), train['value'].to_numpy(),
                                np.array([[test['Lon'], test['Lat']]]), method='nearest')
            if pred is not None and len(pred) > 0 and np.isfinite(pred[0]):
                obs_vals.append(test['value'])
                pred_vals.append(pred[0])

        obs_vals = np.array(obs_vals, dtype=float)
        pred_vals = np.array(pred_vals, dtype=float)
        rows.append({
            'date': dt,
            'n_points': n,
            'rmse': compute_rmse(obs_vals, pred_vals),
            'corr': compute_corr(obs_vals, pred_vals)
        })

    return pd.DataFrame(rows)


def compare_with_era5(df_station: pd.DataFrame, df_era5: pd.DataFrame):
    merged = df_station[['date', 'StationName', 'value', 'season', 'year']].merge(
        df_era5, on=['date', 'StationName'], how='inner'
    )
    if merged.empty:
        empty = pd.DataFrame()
        return empty, empty, empty, empty

    merged['bias'] = merged['value'] - merged['era5_value']
    merged['abs_error'] = merged['bias'].abs()
    merged['sq_error'] = merged['bias'] ** 2

    overall = pd.DataFrame([{
        'n_pairs': len(merged),
        'rmse': compute_rmse(merged['value'].to_numpy(), merged['era5_value'].to_numpy()),
        'corr': compute_corr(merged['value'].to_numpy(), merged['era5_value'].to_numpy()),
        'mean_bias': merged['bias'].mean(),
        'mean_abs_error': merged['abs_error'].mean()
    }])

    seasonal = merged.groupby('season').apply(
        lambda g: pd.Series({
            'n_pairs': len(g),
            'rmse': compute_rmse(g['value'].to_numpy(), g['era5_value'].to_numpy()),
            'corr': compute_corr(g['value'].to_numpy(), g['era5_value'].to_numpy()),
            'mean_bias': g['bias'].mean(),
            'mean_abs_error': g['abs_error'].mean()
        })
    ).reset_index()

    stationwise = merged.groupby('StationName').apply(
        lambda g: pd.Series({
            'n_pairs': len(g),
            'rmse': compute_rmse(g['value'].to_numpy(), g['era5_value'].to_numpy()),
            'corr': compute_corr(g['value'].to_numpy(), g['era5_value'].to_numpy()),
            'mean_bias': g['bias'].mean()
        })
    ).reset_index()

    return merged, overall, seasonal, stationwise


# ======================================
# Plotting
# ======================================
def save_plot(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)


def plot_station_availability(df: pd.DataFrame, var_name: str):
    counts = df.groupby('date')['StationName'].nunique().reset_index(name='station_count')
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(counts['date'], counts['station_count'], lw=1.0)
    ax.set_title(f'{var_name}: Daily Station Availability')
    ax.set_xlabel('Date')
    ax.set_ylabel('Reporting stations')
    ax.grid(alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_station_availability.png')


def plot_yearly_mean(df_yearly: pd.DataFrame, var_name: str, unit: str):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df_yearly['year'], df_yearly['mean'], marker='o', lw=1.2)
    ax.set_title(f'{var_name}: Annual Mean')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'Mean ({unit})')
    ax.grid(alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_annual_mean.png')


def plot_seasonal_means(df_seasonal: pd.DataFrame, var_name: str, unit: str):
    order = ['DJF', 'MAM', 'JJA', 'SON']
    dfp = df_seasonal.copy()
    dfp['season'] = pd.Categorical(dfp['season'], categories=order, ordered=True)
    dfp = dfp.sort_values('season')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dfp['season'], dfp['mean'])
    ax.set_title(f'{var_name}: Seasonal Mean')
    ax.set_xlabel('Season')
    ax.set_ylabel(f'Mean ({unit})')
    ax.grid(axis='y', alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_seasonal_mean.png')


def plot_sample_map(grid_lon: np.ndarray, grid_lat: np.ndarray, grid: np.ndarray,
                    sample_points: pd.DataFrame, var_name: str, unit: str, date_str: str):
    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = ax.pcolormesh(grid_lon, grid_lat, grid, shading='auto', cmap='viridis')
    ax.scatter(sample_points['Lon'], sample_points['Lat'], c='k', s=10, alpha=0.7, label='Stations')
    ax.set_title(f'{var_name}: 2° Interpolated Grid ({date_str})')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(loc='upper right')
    cbar = fig.colorbar(pcm, ax=ax)
    cbar.set_label(unit)
    save_plot(fig, MAP_DIR / f'{var_name}_map_{date_str}.png')


def plot_skill(df_skill: pd.DataFrame, var_name: str):
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(df_skill['date'], df_skill['rmse'], marker='o', ms=3, lw=0.8)
    axes[0].set_ylabel('RMSE')
    axes[0].set_title(f'{var_name}: Leave-One-Out Interpolation Skill')
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_skill['date'], df_skill['corr'], marker='o', ms=3, lw=0.8)
    axes[1].set_ylabel('Correlation')
    axes[1].set_xlabel('Date')
    axes[1].grid(alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_interpolation_skill.png')


def plot_era5_scatter(df_merged: pd.DataFrame, var_name: str, unit: str):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df_merged['era5_value'], df_merged['value'], s=8, alpha=0.35)
    finite = np.isfinite(df_merged['era5_value']) & np.isfinite(df_merged['value'])
    if finite.any():
        mn = float(min(df_merged.loc[finite, 'era5_value'].min(), df_merged.loc[finite, 'value'].min()))
        mx = float(max(df_merged.loc[finite, 'era5_value'].max(), df_merged.loc[finite, 'value'].max()))
        ax.plot([mn, mx], [mn, mx], 'r--', lw=1)
    ax.set_title(f'{var_name}: USCRN vs ERA5')
    ax.set_xlabel(f'ERA5 ({unit})')
    ax.set_ylabel(f'USCRN ({unit})')
    ax.grid(alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_era5_scatter.png')


def plot_era5_seasonal_rmse(df_seasonal: pd.DataFrame, var_name: str):
    order = ['DJF', 'MAM', 'JJA', 'SON']
    dfp = df_seasonal.copy()
    dfp['season'] = pd.Categorical(dfp['season'], categories=order, ordered=True)
    dfp = dfp.sort_values('season')
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dfp['season'], dfp['rmse'])
    ax.set_title(f'{var_name}: Seasonal RMSE vs ERA5')
    ax.set_xlabel('Season')
    ax.set_ylabel('RMSE')
    ax.grid(axis='y', alpha=0.3)
    save_plot(fig, PLOT_DIR / f'{var_name}_era5_seasonal_rmse.png')


# ======================================
# Export helpers
# ======================================
def save_csv(df: pd.DataFrame, path: Path):
    df.to_csv(path, index=False)


def save_grid_csv(grid_lon: np.ndarray, grid_lat: np.ndarray, grid: np.ndarray,
                  date_str: str, var_name: str):
    out = pd.DataFrame({
        'date': date_str,
        'lon': grid_lon.ravel(),
        'lat': grid_lat.ravel(),
        'value': grid.ravel()
    })
    save_csv(out, CSV_DIR / f'{var_name}_grid_{date_str}.csv')


# ======================================
# Main workflow per variable
# ======================================
def process_variable(var_name: str, cfg: dict, station_info: pd.DataFrame):
    print(f'Processing {var_name}...')

    wide = load_variable_data(cfg['file'])
    long_df = reshape_to_long(wide)
    merged = merge_station_coords(long_df, station_info)

    save_csv(merged, CSV_DIR / f'{var_name}_merged_long.csv')
    save_csv(descriptive_stats(merged), CSV_DIR / f'{var_name}_global_stats.csv')
    save_csv(station_stats(merged), CSV_DIR / f'{var_name}_station_stats.csv')

    df_seasonal = seasonal_stats(merged)
    df_yearly = yearly_stats(merged)
    df_ks = seasonal_distribution_tests(merged)

    save_csv(df_seasonal, CSV_DIR / f'{var_name}_seasonal_stats.csv')
    save_csv(df_yearly, CSV_DIR / f'{var_name}_yearly_stats.csv')
    save_csv(df_ks, CSV_DIR / f'{var_name}_seasonal_ks_test.csv')

    plot_station_availability(merged, var_name)
    plot_yearly_mean(df_yearly, var_name, cfg['unit'])
    plot_seasonal_means(df_seasonal, var_name, cfg['unit'])

    _, _, grid_lon, grid_lat = create_grid()
    sample_date = pd.to_datetime(cfg['sample_date'])
    df_day = merged[merged['date'] == sample_date].copy()

    if not df_day.empty:
        grid = interpolate_points(df_day, grid_lon, grid_lat, method='linear')
        date_str = sample_date.strftime('%Y-%m-%d')
        save_grid_csv(grid_lon, grid_lat, grid, date_str, var_name)
        plot_sample_map(grid_lon, grid_lat, grid, df_day, var_name, cfg['unit'], date_str)

    df_skill = interpolation_skill_loo(merged, method='linear', max_dates=SKILL_SAMPLE_DATES)
    save_csv(df_skill, CSV_DIR / f'{var_name}_interpolation_skill.csv')
    plot_skill(df_skill, var_name)

    era5_summary_row = {
        'era5_pairs': np.nan,
        'era5_rmse': np.nan,
        'era5_corr': np.nan,
        'era5_mean_bias': np.nan
    }

    df_era5 = load_era5_if_available(cfg['era5_file'])
    if df_era5 is not None:
        df_cmp, df_era5_overall, df_era5_seasonal, df_era5_station = compare_with_era5(merged, df_era5)

        if not df_cmp.empty:
            save_csv(df_cmp, CSV_DIR / f'{var_name}_era5_matched_pairs.csv')
            save_csv(df_era5_overall, CSV_DIR / f'{var_name}_era5_overall_comparison.csv')
            save_csv(df_era5_seasonal, CSV_DIR / f'{var_name}_era5_seasonal_comparison.csv')
            save_csv(df_era5_station, CSV_DIR / f'{var_name}_era5_station_comparison.csv')
            plot_era5_scatter(df_cmp, var_name, cfg['unit'])
            plot_era5_seasonal_rmse(df_era5_seasonal, var_name)

            era5_summary_row = {
                'era5_pairs': float(df_era5_overall.iloc[0]['n_pairs']),
                'era5_rmse': float(df_era5_overall.iloc[0]['rmse']),
                'era5_corr': float(df_era5_overall.iloc[0]['corr']),
                'era5_mean_bias': float(df_era5_overall.iloc[0]['mean_bias'])
            }

    summary = pd.DataFrame([{
        'variable': var_name,
        'records': len(merged),
        'stations': merged['StationName'].nunique(),
        'start_date': merged['date'].min(),
        'end_date': merged['date'].max(),
        'overall_mean': merged['value'].mean(),
        'overall_std': merged['value'].std(),
        'mean_skill_rmse': df_skill['rmse'].mean(),
        'mean_skill_corr': df_skill['corr'].mean(),
        **era5_summary_row,
        'era5_note': cfg['era5_unit_note']
    }])
    save_csv(summary, CSV_DIR / f'{var_name}_summary.csv')
    return summary


# ======================================
# Run all variables
# ======================================
def main():
    station_info = load_station_info(STATION_INFO_FILE)
    summaries = []

    for var_name, cfg in VARIABLE_CONFIGS.items():
        if cfg['file'].exists():
            summaries.append(process_variable(var_name, cfg, station_info))
        else:
            print(f'Skipped {var_name}: file not found -> {cfg["file"].name}')

    if summaries:
        final_summary = pd.concat(summaries, ignore_index=True)
        save_csv(final_summary, CSV_DIR / 'all_variables_summary.csv')
        print(final_summary)


if __name__ == '__main__':
    main()